# 10 — GPT-5-mini (Azure OpenAI)

Primo dei notebook (10–12) che valutano **LLM cloud su Azure AI Foundry** con lo stesso prompt
zero-shot dei notebook locali 07–09 (prompt in inglese, documento passato da
`prepara_documento`; senza il prefisso `/no_think`, artefatto specifico dei modelli Qwen).

**Deviazioni deliberate dal protocollo 07–09** (GPT-5-mini è un modello con *reasoning*; la
famiglia gpt-4o-mini è ritirata da Azure e non è più deployabile):

- niente `temperature` (i modelli GPT-5 accettano solo il default 1);
- `max_completion_tokens=1500` con `reasoning_effort='minimal'` al posto di `max_tokens=300`:
  i token di reasoning consumano il budget di completamento **prima** della risposta visibile,
  esattamente il caso gemma del notebook 08 (con 300 la risposta resterebbe spesso vuota).
  La lunghezza effettiva dei riassunti resta osservabile nella metrica `parole_generate`.

Due ambiti:

- `SCOPE='sample'` — il campione condiviso da 100 esempi (confronto con tutti i metodi 01–09,
  costo di pochi centesimi);
- `SCOPE='test'` — l'intera split **test** pulita di `complete.tab` (5.610 righe, ~8 $ con
  GPT-5-mini): confronto pulito senza le avvertenze di leakage della split train.

Prerequisiti: un deployment **Global Standard** di `gpt-5-mini` nella risorsa Azure AI Foundry e
le variabili d'ambiente `AZURE_OPENAI_ENDPOINT` / `AZURE_OPENAI_API_KEY` (mai chiavi nel codice).
La corsa sull'intero dataset (56.101 righe) usa invece la **Batch API** nel notebook 13.

## Ripresa e rischio di mescolare corse

⚠️ Il ciclo condiviso salta i `row_id` già presenti nel TSV di output: rieseguire la generazione
su un file esistente **aggiunge solo le righe mancanti**. È il comportamento voluto per riprendere
una corsa interrotta, ma con un modello, un deployment o una configurazione diversi si
mescolerebbero due corse nello stesso file: in quel caso **eliminare prima** il TSV e rigenerare
tutto (rieseguendo poi la valutazione). Ogni ambito (`sample`, `test`) scrive su un file separato.

In [ ]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401
except ImportError:
    %pip install pyAutoSummarizer
try:
    import openai  # noqa: F401
except ImportError:
    %pip install openai

In [ ]:
# --- Configurazione ---------------------------------------------------------
import os
import summ_utils as su

METODO     = 'gpt5mini'
SCOPE      = 'sample'    # 'sample' = campione condiviso; 'test' = intera split test (5.610 righe)
N_SAMPLES  = 100         # deve combaciare con il campione creato dal notebook 00
SEED       = 42
LIMIT      = None        # es. 3 per uno smoke test rapido; None = tutti

MODELLO     = 'gpt-5-mini'
DEPLOYMENT  = 'gpt-5-mini'           # nome del deployment Global Standard nel portale Azure
AZURE_ENDPOINT = os.environ['AZURE_OPENAI_ENDPOINT']   # es. https://<risorsa>.openai.azure.com/
AZURE_API_KEY  = os.environ['AZURE_OPENAI_API_KEY']
# API "v1" di Azure OpenAI: nessuna api-version datata (le vecchie preview sono state ritirate)
BASE_URL = AZURE_ENDPOINT.rstrip('/') + '/openai/v1/'

# GPT-5-mini e' un modello con reasoning: niente temperature (solo default) e budget
# max_completion_tokens condiviso con i token di reasoning -> 1500 come gemma (notebook 08)
MAX_COMPLETION_TOKENS = 1500
REASONING_EFFORT      = 'minimal'
PROMPT_SYSTEM = ('You are a helpful assistant that summarizes news articles '
                 'from different sources concisely.')
PROMPT_USER   = 'Summarize the following document into a comprehensive summary: {documento}'
ETICHETTA   = 'GPT-5-mini '
NOTE_CONFIG = ('prompt zero-shot in inglese identico ai notebook 07-09 (senza /no_think); '
               'modello con reasoning: niente temperature, max_completion_tokens=1500 con '
               "reasoning_effort='minimal' (caso gemma); ambiti sample e test")

BASE = su.trova_base_dir()
P    = su.percorsi_standard(BASE)
SAMPLE_PATH = P['sample_dir'] / f'sample_{N_SAMPLES}_seed{SEED}.tsv'
OUT_PATH    = P['summaries_dir'] / f'{METODO}_{SCOPE}.tsv'

config = {'modello': MODELLO, 'deployment': DEPLOYMENT,
          'backend': 'Azure OpenAI (chat completions, API v1)',
          'max_completion_tokens': MAX_COMPLETION_TOKENS,
          'reasoning_effort': REASONING_EFFORT,
          'prompt_system': PROMPT_SYSTEM, 'prompt_user': PROMPT_USER,
          'note': NOTE_CONFIG}

print(f'Deployment : {DEPLOYMENT} via {BASE_URL}')
print(f'Ambito     : {SCOPE}')
print(f'Output     : {OUT_PATH}')

## Generazione dei riassunti

Il client è `openai.OpenAI` puntato alla rotta **v1** di Azure OpenAI
(`https://<risorsa>.openai.azure.com/openai/v1/`): stessa interfaccia chat-completions dei
notebook 07–09, senza `api-version` datata (le vecchie preview sono state ritirate da Azure), ma
`model=` è il **nome del deployment** Azure, non il nome del modello, e i parametri sono quelli
dei modelli con reasoning (vedi sopra). Il ciclo e la scrittura incrementale (con ripresa) sono
quelli condivisi di `summ_utils`; una risposta vuota (per esempio budget esaurito dal reasoning,
`finish_reason=length`) solleva un'eccezione, così la riga viene registrata come errore e **non**
scritta nel TSV (ritentabile alla corsa successiva).

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=AZURE_API_KEY)

def genera(documento):
    risposta = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=[{'role': 'system', 'content': PROMPT_SYSTEM},
                  {'role': 'user', 'content': PROMPT_USER.format(documento=documento)}],
        max_completion_tokens=MAX_COMPLETION_TOKENS,
        reasoning_effort=REASONING_EFFORT)
    scelta = risposta.choices[0]
    contenuto = scelta.message.content
    if not contenuto or not contenuto.strip():
        # solleva -> il ciclo condiviso registra l'errore e NON scrive la riga
        raise RuntimeError(f'risposta vuota (finish_reason={scelta.finish_reason})')
    return contenuto.strip()

if SCOPE == 'sample':
    esempi = su.carica_campione(SAMPLE_PATH)
elif SCOPE == 'test':
    esempi = su.itera_split(P['complete_tab'], 'test')
else:
    raise ValueError(f'SCOPE non valido: {SCOPE!r}')

scrittore = su.ScrittoreRiassunti(OUT_PATH)
errori = su.ciclo_summarization(esempi, scrittore, genera, limit=LIMIT,
                                etichetta=ETICHETTA)
scrittore.chiudi()

## Valutazione (indipendente dalla generazione)

Legge **solo** i file salvati; rieseguibile senza rigenerare i riassunti. Metriche ROUGE-1/2/L
(F1, precisione, recall), BLEU e METEOR con normalizzazione identica per tutti i metodi del
benchmark. Output: `results/metrics/{metodo}_{scope}_per_example.csv` e `…_aggregate.json`.
Per l'ambito `test` i riferimenti vengono letti in streaming da `complete.tab`.

In [ ]:
import json

riassunti = su.carica_riassunti(OUT_PATH)
if SCOPE == 'sample':
    riferimenti = su.carica_campione(SAMPLE_PATH)
else:
    riferimenti = su.itera_split(P['complete_tab'], 'test')

righe, aggregato = su.valuta_e_salva(riferimenti, riassunti, METODO, SCOPE,
                                     P['metrics_dir'], config)
print(json.dumps(aggregato['overall'], indent=2))
print('\nMedie per split:')
for split, valori in aggregato['per_split'].items():
    print(f"  {split:5s} (n={valori['n_esempi']}): ROUGE-1 F1 = {valori['rouge1_f1']:.3f}")

## Ispezione qualitativa

In [ ]:
if SCOPE == 'sample':
    riferimenti = su.carica_campione(SAMPLE_PATH)
else:
    riferimenti = su.itera_split(P['complete_tab'], 'test')
su.mostra_esempi(riferimenti, riassunti, quanti=2)